In [1]:
print("Hello World")

Hello World


In [2]:
from pathlib import Path
from datetime import datetime
import json
import os
import textwrap

import numpy as np
import pandas as pd
from groq import Groq


In [ ]:
ROOT = Path.cwd()
ARTIFACTS_DIR = ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

In [ ]:
GROQ_API_KEY = "yourapikey"
GROQ_MODEL = "llama-3.3-70b-versatile"

if GROQ_API_KEY.strip():
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY.strip()

if not os.getenv("GROQ_API_KEY"):
    print("Add your Groq API key before running the generation cell.")
else:
    print("Groq API key found.")

Groq API key found.


In [6]:
race_data = {
    "event": {
        "season": 2025,
        "round": 5,
        "race_name": "Saudi Arabian Grand Prix",
        "circuit": "Jeddah Corniche Circuit",
        "date": "2025-04-20",
        "total_laps": 50,
        "weather": "Dry, warm evening conditions",
        "safety_car_laps": [1, 2, 3],
        "virtual_safety_car_laps": [],
    },
    "race_notes": [
        "Early safety car compressed the field and opened the window for teams to adjust tire plans.",
        "Track position was valuable because overtaking required a strong DRS run into Turn 1.",
        "Medium-to-hard one-stop strategies were preferred by most front-running teams.",
    ],
    "results": [
        {"position": 1, "driver": "Oscar Piastri", "team": "McLaren", "grid": 2, "points": 25, "status": "Finished", "fastest_lap": "1:31.778"},
        {"position": 2, "driver": "Max Verstappen", "team": "Red Bull Racing", "grid": 1, "points": 18, "status": "Finished", "fastest_lap": "1:31.986"},
        {"position": 3, "driver": "Charles Leclerc", "team": "Ferrari", "grid": 4, "points": 15, "status": "Finished", "fastest_lap": "1:32.104"},
        {"position": 4, "driver": "Lando Norris", "team": "McLaren", "grid": 10, "points": 12, "status": "Finished", "fastest_lap": "1:31.914"},
        {"position": 5, "driver": "George Russell", "team": "Mercedes", "grid": 3, "points": 10, "status": "Finished", "fastest_lap": "1:32.301"},
        {"position": 6, "driver": "Andrea Kimi Antonelli", "team": "Mercedes", "grid": 5, "points": 8, "status": "Finished", "fastest_lap": "1:32.487"},
        {"position": 7, "driver": "Lewis Hamilton", "team": "Ferrari", "grid": 7, "points": 6, "status": "Finished", "fastest_lap": "1:32.533"},
        {"position": 8, "driver": "Carlos Sainz", "team": "Williams", "grid": 8, "points": 4, "status": "Finished", "fastest_lap": "1:32.710"},
        {"position": 9, "driver": "Alex Albon", "team": "Williams", "grid": 11, "points": 2, "status": "Finished", "fastest_lap": "1:32.855"},
        {"position": 10, "driver": "Isack Hadjar", "team": "Racing Bulls", "grid": 12, "points": 1, "status": "Finished", "fastest_lap": "1:33.012"},
    ],
    "strategies": [
        {"driver": "Oscar Piastri", "team": "McLaren", "stops": 1, "stints": ["Medium 20 laps", "Hard 30 laps"], "summary": "Used clean air after the stop to protect tire life and control the closing phase."},
        {"driver": "Max Verstappen", "team": "Red Bull Racing", "stops": 1, "stints": ["Medium 19 laps", "Hard 31 laps"], "summary": "Led early but lost flexibility after the safety car and had limited tire advantage late."},
        {"driver": "Charles Leclerc", "team": "Ferrari", "stops": 1, "stints": ["Medium 21 laps", "Hard 29 laps"], "summary": "Extended the first stint and gained pace as the hard tire stabilized."},
        {"driver": "Lando Norris", "team": "McLaren", "stops": 1, "stints": ["Hard 33 laps", "Medium 17 laps"], "summary": "Alternative strategy helped recover from a lower grid slot with strong late-race pace."},
        {"driver": "George Russell", "team": "Mercedes", "stops": 1, "stints": ["Medium 18 laps", "Hard 32 laps"], "summary": "Stayed in the lead group early but faded slightly on the final stint."},
        {"driver": "Andrea Kimi Antonelli", "team": "Mercedes", "stops": 1, "stints": ["Medium 18 laps", "Hard 32 laps"], "summary": "Managed a clean race and protected points under pressure."},
        {"driver": "Lewis Hamilton", "team": "Ferrari", "stops": 1, "stints": ["Medium 17 laps", "Hard 33 laps"], "summary": "Held position but lacked enough straight-line advantage to attack Mercedes."},
        {"driver": "Carlos Sainz", "team": "Williams", "stops": 1, "stints": ["Medium 19 laps", "Hard 31 laps"], "summary": "Converted grid position into points with disciplined tire management."},
        {"driver": "Alex Albon", "team": "Williams", "stops": 1, "stints": ["Hard 34 laps", "Medium 16 laps"], "summary": "Late medium stint helped secure a double-points result for Williams."},
        {"driver": "Isack Hadjar", "team": "Racing Bulls", "stops": 1, "stints": ["Medium 20 laps", "Hard 30 laps"], "summary": "Stayed out of trouble and took the final point."},
    ],
    "incidents": [
        {"lap": 1, "description": "Opening-lap incident triggered an early safety car."},
        {"lap": 20, "description": "Front-runners began the main pit cycle."},
        {"lap": 34, "description": "Drivers on long first stints switched to softer rubber for late-race attacks."},
    ],
}

In [7]:
results_df = pd.DataFrame(race_data["results"])
strategies_df = pd.DataFrame(race_data["strategies"])
incidents_df = pd.DataFrame(race_data["incidents"])


In [8]:
results_df["position_gain"] = results_df["grid"] - results_df["position"]
results_df["classified"] = results_df["status"].str.lower().eq("finished")

analysis_df = results_df.merge(
    strategies_df[["driver", "stops", "stints", "summary"]],
    on="driver",
    how="left",
)


In [9]:

team_df = (
    analysis_df.groupby("team", as_index=False)
    .agg(
        best_finish=("position", "min"),
        total_points=("points", "sum"),
        avg_finish=("position", "mean"),
        net_position_gain=("position_gain", "sum"),
        drivers=("driver", lambda values: ", ".join(values)),
    )
    .sort_values(["total_points", "best_finish"], ascending=[False, True])
)

analysis_df

,position,driver,team,grid,points,status,fastest_lap,position_gain,classified,stops,stints,summary
0,1,Oscar Piastri,McLaren,2,25,Finished,1:31.778,1,True,1,"[Medium 20 laps, Hard 30 laps]",Used clean air after the stop to protect tire life and control the closing phase.
1,2,Max Verstappen,Red Bull Racing,1,18,Finished,1:31.986,-1,True,1,"[Medium 19 laps, Hard 31 laps]",Led early but lost flexibility after the safety car and had limited tire advantage late.
2,3,Charles Leclerc,Ferrari,4,15,Finished,1:32.104,1,True,1,"[Medium 21 laps, Hard 29 laps]",Extended the first stint and gained pace as the hard tire stabilized.
3,4,Lando Norris,McLaren,10,12,Finished,1:31.914,6,True,1,"[Hard 33 laps, Medium 17 laps]",Alternative strategy helped recover from a lower grid slot with strong late-race pace.
4,5,George Russell,Mercedes,3,10,Finished,1:32.301,-2,True,1,"[Medium 18 laps, Hard 32 laps]",Stayed in the lead group early but faded slightly on the final stint.
5,6,Andrea Kimi Antonelli,Mercedes,5,8,Finished,1:32.487,-1,True,1,"[Medium 18 laps, Hard 32 laps]",Managed a clean race and protected points under pressure.
6,7,Lewis Hamilton,Ferrari,7,6,Finished,1:32.533,0,True,1,"[Medium 17 laps, Hard 33 laps]",Held position but lacked enough straight-line advantage to attack Mercedes.
7,8,Carlos Sainz,Williams,8,4,Finished,1:32.710,0,True,1,"[Medium 19 laps, Hard 31 laps]",Converted grid position into points with disciplined tire management.
8,9,Alex Albon,Williams,11,2,Finished,1:32.855,2,True,1,"[Hard 34 laps, Medium 16 laps]",Late medium stint helped secure a double-points result for Williams.
9,10,Isack Hadjar,Racing Bulls,12,1,Finished,1:33.012,2,True,1,"[Medium 20 laps, Hard 30 laps]",Stayed out of trouble and took the final point.


In [10]:
team_df

,team,best_finish,total_points,avg_finish,net_position_gain,drivers
1,McLaren,1,37,2.5,7,"Oscar Piastri, Lando Norris"
0,Ferrari,3,21,5.0,1,"Charles Leclerc, Lewis Hamilton"
4,Red Bull Racing,2,18,2.0,-1,Max Verstappen
2,Mercedes,5,18,5.5,-3,"George Russell, Andrea Kimi Antonelli"
5,Williams,8,6,8.5,2,"Carlos Sainz, Alex Albon"
3,Racing Bulls,10,1,10.0,2,Isack Hadjar


In [11]:
def format_position_gain(value):
    if value > 0:
        return f"gained {value}"
    if value < 0:
        return f"lost {abs(value)}"
    return "held position"

In [12]:

def build_race_context(race_data, analysis_df, team_df, incidents_df):
    event = race_data["event"]
    result_lines = []
    for row in analysis_df.sort_values("position").itertuples(index=False):
        stints = ", ".join(row.stints) if isinstance(row.stints, list) else "not supplied"
        result_lines.append(
            f"P{row.position}: {row.driver} ({row.team}), started P{row.grid}, "
            f"{format_position_gain(row.position_gain)}, {row.points} pts, "
            f"strategy: {row.stops} stop(s), stints: {stints}. Note: {row.summary}"
        )

    team_lines = []
    for row in team_df.itertuples(index=False):
        team_lines.append(
            f"{row.team}: {row.total_points} pts, best finish P{row.best_finish}, "
            f"avg finish {row.avg_finish:.1f}, net grid gain {row.net_position_gain}, drivers: {row.drivers}"
        )

    incident_lines = [f"Lap {row.lap}: {row.description}" for row in incidents_df.itertuples(index=False)]
    notes = race_data.get("race_notes", [])

    context = {
        "event": event,
        "race_notes": notes,
        "results": result_lines,
        "teams": team_lines,
        "incidents": incident_lines,
    }
    return json.dumps(context, indent=2)


In [13]:
race_context = build_race_context(race_data, analysis_df, team_df, incidents_df)
print(race_context[:3000])

{
  "event": {
    "season": 2025,
    "round": 5,
    "race_name": "Saudi Arabian Grand Prix",
    "circuit": "Jeddah Corniche Circuit",
    "date": "2025-04-20",
    "total_laps": 50,
    "weather": "Dry, warm evening conditions",
    "safety_car_laps": [
      1,
      2,
      3
    ],
    "virtual_safety_car_laps": []
  },
  "race_notes": [
    "Early safety car compressed the field and opened the window for teams to adjust tire plans.",
    "Track position was valuable because overtaking required a strong DRS run into Turn 1.",
    "Medium-to-hard one-stop strategies were preferred by most front-running teams."
  ],
  "results": [
    "P1: Oscar Piastri (McLaren), started P2, gained 1, 25 pts, strategy: 1 stop(s), stints: Medium 20 laps, Hard 30 laps. Note: Used clean air after the stop to protect tire life and control the closing phase.",
    "P2: Max Verstappen (Red Bull Racing), started P1, lost 1, 18 pts, strategy: 1 stop(s), stints: Medium 19 laps, Hard 31 laps. Note: Led ea

In [15]:
SYSTEM_PROMPT = """
You are an expert Formula 1 race analyst writing for a data-driven motorsport report.
Use only the race facts supplied by the user. Do not invent lap times, radio messages, penalties, tire data, or weather details.
Write clearly, analytically, and confidently.
""".strip()

USER_PROMPT_TEMPLATE = """
Generate a post-race Formula 1 report from this structured race data.

Required output sections:
1. Race Report
2. Driver Analysis
3. Team Analysis
4. Strategy Takeaways
5. One-Sentence Summary

Writing requirements:
- Mention the race name, winner, podium, and major strategic theme.
- Explain position gains/losses where they matter.
- Include at least three driver-specific insights.
- Include at least three team-specific insights.
- Keep the report grounded in the supplied facts.
- Use polished motorsport journalism style, not bullet-only notes.

Race data:
{race_context}
""".strip()

prompt = USER_PROMPT_TEMPLATE.format(race_context=race_context)

In [16]:
def generate_race_report(prompt, model=GROQ_MODEL, temperature=0.35, max_tokens=1800):
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError("Missing GROQ_API_KEY. Add it in the API key cell before running this cell.")

    client = Groq(api_key=api_key)
    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return completion.choices[0].message.content

race_report = generate_race_report(prompt)
print(race_report)

## 1. Race Report
The Saudi Arabian Grand Prix, held under dry and warm evening conditions at the Jeddah Corniche Circuit, saw a captivating display of strategic prowess and driving skill. The race was marked by an early safety car period, which significantly compressed the field and opened a window for teams to adjust their tire plans. This strategic theme played out throughout the race, with most front-running teams opting for a medium-to-hard one-stop strategy. Oscar Piastri, starting from second, capitalized on this scenario to claim the top step of the podium, gaining one position over the course of the race. Max Verstappen, who started from pole, finished second, losing the lead due to the strategic implications of the safety car and limited tire advantage in the closing stages. Charles Leclerc secured the third spot, gaining one position from his starting fourth place, by extending his first stint and finding pace as the hard tire stabilized.

## 2. Driver Analysis
Oscar Piastri

In [17]:
event = race_data["event"]
slug = f"{event['season']}_{event['race_name'].lower().replace(' ', '_')}_race_report.md"
report_path = ARTIFACTS_DIR / slug

report_markdown = f"""# {event['season']} {event['race_name']} - AI Race Report

Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Model: {GROQ_MODEL}

{race_report}
"""

report_path.write_text(report_markdown, encoding="utf-8")
print(f"Saved report to: {report_path}")

Saved report to: /Users/santhoshkumarv/vs_code_projects/internship-harshith/projects/capstone_project/artifacts/2025_saudi_arabian_grand_prix_race_report.md
